In [0]:
landing_path="/Volumes/pricing_data/deltalake/landing/order_details/"
bronze_path="/Volumes/pricing_data/deltalake/bronze/order_details/"
checkpoint_path="/Volumes/pricing_data/deltalake/bronze/checkpoint/"

In [0]:
try:
    processed_files_df=spark.read.format("delta").load(checkpoint_path)
    processed_files=[f.file_name for f in processed_files_df.collect()]
except :
    processed_files=[]

In [0]:
landing_files=dbutils.fs.ls(landing_path)

unprocessed_files=[fileinfo.path for fileinfo in landing_files if fileinfo.name not in processed_files]

In [0]:
from pyspark.sql.functions import current_timestamp

df=spark.read.format("parquet").load(unprocessed_files)
df=df.withColumn("file_ingestion_date", current_timestamp()).withColumn("file_updated_date", current_timestamp())
df.write.format("delta").mode("append").save(bronze_path)

In [0]:
from pyspark.sql import Row
current_processed_files=[Row(f.split('/')[-1]) for f in unprocessed_files]

file_df=spark.createDataFrame(current_processed_files)

file_df=file_df.withColumnRenamed("_1", "file_name")

file_df.write.format("delta").mode("append").save(checkpoint_path)